🎯 What is Hyperparameter Tuning in Neural Networks?

Neural Networks are very flexible, but that flexibility creates a problem:

👉 Too many choices

How many hidden layers?

How many neurons per layer?

Which activation function?

Learning rate?

Optimizer?

Weight initialization?

There is no formula that tells you the perfect values.
So the only practical solution is:

Try many combinations and see which one performs best on validation data.

🔁 How Do We Try Many Combinations?

Two common approaches (you already know these from ML):

GridSearchCV – try all combinations

RandomizedSearchCV – try random combinations

⚠️ Problem:
Scikit-Learn does not directly understand Keras models

✅ Solution:
Wrap Keras models so they behave like Scikit-Learn models

# Step1 - Create a Model-Building Function

Instead of directly writing a model, we create a function that:

Accepts hyperparameters

Builds the model

Compiles it

Returns it

In [2]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense , Flatten

D:\JUPYTER NOTEBOOK\DEEP LEARNING\dl_env\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [7]:
def build_model(n_hidden = 1 , n_neurons = 30 , learning_rate = 0.003):
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(8,)))

    for _ in range(n_hidden):
        model.add(Dense(n_neurons , activation = 'relu'))

    model.add(keras.layers.Dense(1))

    optimizer = keras.optimizers.SGD(learning_rate = learning_rate)
    model.compile(loss = 'mse' , optimizer = optimizer)

    return model
                  

🔍 What this function does:

n_hidden → number of hidden layers

n_neurons → neurons per hidden layer

learning_rate → SGD learning rate

input_shape → number of input features

Output layer has 1 neuron → regression task

Loss = MSE (Mean Squared Error)

📌 Important trick
The options dictionary ensures input_shape is passed only to the first layer.

# 🧩 Step2: Wrap the Model with ]SciKeras

Now we convert this Keras model into a Scikit-Learn-style estimator:

In [9]:
from scikeras.wrappers import KerasRegressor
keras_reg = KerasRegressor(
    model = build_model ,
    epochs = 100,
    batch_size = 32, 
    verbose = 0
)


Now this object behaves like:

LinearRegression

RandomForestRegressor

You can call:

.fit()

.score()

.predict()

# Step -3 Fit , Score  , Predict (Same as Skleran)

In [12]:
keras_reg.fit(
    X_train , y_train,
    validation_data = (X_valid , y_valid),
    callbacks = [keras.callbacks.EarlyStopping(patience=10)]
    
)
mse_Test = keras_reg.score(X_test , y_test)
y_pred = keras_reg.predict(X_new)

NameError: name 'X_train' is not defined

In [13]:
param_grid = {
    'model__n_hidden': [1,2,3],
    'model__n_neurons': [20 , 30, 50],
    'model__learning_rate':[0.01,0.001]
}
grid = GridSearchCV(keras_reg, param_grid , cv = 3)
grid.fit(X_train , y_train)

NameError: name 'GridSearchCV' is not defined

In [10]:
# pip install scikeras